# MotionGPT: Milestone 2 (pretrained inference in Colab)

Run the cells **top to bottom**. Before starting: **Runtime > Change runtime type > T4 GPU**.

Repo: https://github.com/OpenMotionLab/MotionGPT  |  Paper: https://arxiv.org/abs/2306.14795

Take a screenshot of each cell output as you go. Those become your slides.

## 1. Environment check (screenshot this: slide 'Colab Environment')

In [ ]:
import sys, subprocess
print("Python:", sys.version)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
import torch
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. (Optional) Mount Google Drive
Colab resets often. Mount Drive so you can save outputs at the end. Skip if you do not want it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone the repo

In [ ]:
%cd /content
!git clone https://github.com/OpenMotionLab/MotionGPT.git
%cd /content/MotionGPT
!ls

## 4. Install dependencies

The repo was tested on **Python 3.10.6 / PyTorch 2.0.0**; Colab is usually newer, so version conflicts are likely.
If `pip install` fails on a pinned package, note the error (it goes on your 'Challenges' slide), loosen that pin in
`requirements.txt`, and re-run. If it becomes a time sink, try `condacolab` to get a Python 3.10 environment.

In [ ]:
!pip install -r requirements.txt
!python -m spacy download en_core_web_sm

## 5. Download dependencies and pretrained models

These scripts come from the README. They pull from Google Drive / Hugging Face. If one fails (Drive quota errors are
common), use the manual links in the README and put the files where the scripts would have put them.

In [ ]:
!bash prepare/download_smpl_model.sh
!bash prepare/prepare_t5.sh
!bash prepare/download_pretrained_models.sh
# Only needed for evaluation, not for the demo:
# !bash prepare/download_t2m_evaluators.sh

In [ ]:
# Confirm the downloads landed (folder names may differ slightly; adjust if needed)
!ls deps 2>/dev/null
!ls checkpoints 2>/dev/null
!ls demos

## 6. Prompts

Look at the example prompt file first, then write your own in the same format.

In [ ]:
!cat demos/t2m.txt

In [ ]:
prompts = [
    "a person walks forward and then turns around.",
    "a person waves with their right hand.",
    "a person jumps up and down.",
    "a person walks in a circle.",
    "a person kicks with their left leg.",
]
with open("demos/my_prompts.txt", "w") as f:
    f.write("\n".join(prompts) + "\n")
print(open("demos/my_prompts.txt").read())

## 7. Run inference: text-to-motion (the core Milestone 2 result)

In [ ]:
import time
T0 = time.time()   # used later to find only the files this run created

In [ ]:
%%time
!python demo.py --cfg ./configs/config_h3d_stage3.yaml --example ./demos/my_prompts.txt

## 8. Locate the generated motions

The README says outputs are `.npy` files with shape (nframe, 22, 3). The output folder is `TEST.FOLDER` in
`configs/assets.yaml`.

In [ ]:
import glob, os
import numpy as np

found = []
for p in glob.glob("**/*.npy", recursive=True):
    if os.path.getmtime(p) < T0 - 1:
        continue
    try:
        a = np.load(p)
    except Exception:
        continue
    if a.ndim == 4:
        a = a[0]
    if a.ndim == 3 and a.shape[1:] == (22, 3):
        found.append(p)
found.sort()
for p in found:
    print(p, np.load(p).shape)

## 9. Skeleton animation (matplotlib, no Blender needed)

Turns each `.npy` into a GIF you can drop straight into slides.

In [ ]:
import numpy as np, os
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display

# HumanML3D 22-joint skeleton chains
CHAINS = [[0, 2, 5, 8, 11], [0, 1, 4, 7, 10], [0, 3, 6, 9, 12, 15],
          [9, 14, 17, 19, 21], [9, 13, 16, 18, 20]]
COLORS = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

def animate(npy_path, out_path, fps=20, title=""):
    d = np.load(npy_path)
    if d.ndim == 4:
        d = d[0]
    n = d.shape[0]
    # axes: x, z horizontal; y is up in HumanML3D
    mins, maxs = d.reshape(-1, 3).min(0), d.reshape(-1, 3).max(0)
    center = (mins + maxs) / 2
    r = (maxs - mins).max() / 2 + 0.3
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="3d")

    def update(i):
        ax.cla()
        ax.set_xlim(center[0] - r, center[0] + r)
        ax.set_ylim(center[2] - r, center[2] + r)
        ax.set_zlim(center[1] - r, center[1] + r)
        ax.view_init(elev=15, azim=-70)
        ax.set_title(f"{title}\nframe {i + 1}/{n}", fontsize=8)
        for chain, c in zip(CHAINS, COLORS):
            ax.plot(d[i, chain, 0], d[i, chain, 2], d[i, chain, 1], color=c, linewidth=3, marker="o", markersize=3)

    anim = FuncAnimation(fig, update, frames=n, interval=1000 / fps)
    anim.save(out_path, writer=PillowWriter(fps=fps))
    plt.close(fig)

os.makedirs("/content/animations", exist_ok=True)
gifs = []
for i, p in enumerate(found):
    label = prompts[i] if i < len(prompts) else os.path.basename(p)
    out = f"/content/animations/motion_{i}.gif"
    animate(p, out, title=label)
    gifs.append(out)
    print(label, "->", out)

for g in gifs:
    display(Image(filename=g))

## 10. (Optional) Other tasks: m2t, pred, inbetween

The demo's `--task` flag accepts `t2m`, `m2t`, `pred`, `inbetween`. Check `!ls demos` above for example input files.
Uncomment and run what exists. Even one extra task makes a good slide ('one model, different instructions').

In [ ]:
# !python demo.py --cfg ./configs/config_h3d_stage3.yaml --task m2t --example ./demos/m2t.txt
# !python demo.py --cfg ./configs/config_h3d_stage3.yaml --task pred --example ./demos/pred.txt

## 11. Save your evidence
Zips the animations, the motion files and the prompts, then downloads them (and copies to Drive if mounted).

In [ ]:
import shutil, os
os.makedirs("/content/evidence", exist_ok=True)
for p in found:
    shutil.copy(p, "/content/evidence/")
shutil.copy("demos/my_prompts.txt", "/content/evidence/")
shutil.copytree("/content/animations", "/content/evidence/animations", dirs_exist_ok=True)
shutil.make_archive("/content/motiongpt_evidence", "zip", "/content/evidence")

from google.colab import files
files.download("/content/motiongpt_evidence.zip")

if os.path.exists("/content/drive/MyDrive"):
    shutil.copy("/content/motiongpt_evidence.zip", "/content/drive/MyDrive/")
    print("Also copied to Google Drive.")

## Slide checklist

- Environment screenshot (section 1)
- Each error you hit and the fix (for the 'Challenges' slide)
- Prompt -> output file shape and generation time (sections 7 and 8)
- 3 to 5 GIFs (section 9)
- Repo link and README with these exact steps